# Basic Decision Tree from Scratch - Classification (Gini)

In this notebook, we are going to build a decision tree from scratch using median as a cutoff point. 

## Dataset

Dataset: This is a makeup dataset that describe how much study hours a student put in and whether if they will either pass or fail their exam

| Study Hours  | Pass Exam |
| ----- | ----- |
| 1    | No    |
| 2    | No    |
| 3    | No    |
| 4    | No    |
| 5    | Yes   |
| 6    | Yes   |
| 7    | Yes   |
| 8    | Yes   |
| 9    | Yes   |
| 10    | Yes   |
| 11    | Yes   |

In [63]:
import pandas as pd
import json

In [64]:
df = pd.DataFrame({'studied': [1,2,3,4,5,6,7,8,9,10,11,12], 
        'passed': [0,0,0,0,1,1,1,1,1,1,1,1]})

In [65]:
df

,studied,passed
0,1,0
1,2,0
2,3,0
3,4,0
4,5,1
5,6,1
6,7,1
7,8,1
8,9,1
9,10,1


## Gini Formula

### Gini Impurity Formula

For a node:

$$\text{Gini} = 1 - \sum_{k=1}^{K} p_k^2$$


Where:

- K = number of classes
- $p_k$ = proportion of class (k) in the node


#### Binary classification Example (0/1 case)

$$\text{Gini} = 1 - (p_0^2 + p_1^2)$$

Where:

- $p_0 = \frac{0}{N}$
- $p_1 = \frac{1}{N}$

**Intuition**

| Case           | Gini                          |
| -------------- | ----------------------------- |
| all same class | 0 (pure)                      |
| 50/50 split    | 0.5 (max impurity for binary) |

---

### Weighted Gini (for a split)

When you split a node into LEFT and RIGHT:

$$ \text{Weighted Gini} = \frac{N_L}{N} \cdot Gini(L) + \frac{N_R}{N} \cdot Gini(R)$$

Where:

- $N_L$ = number of samples in left node
- $N_R$ = number of samples in right node
- $N = N_L + N_R$


**Intuition**

Weighted Gini means:

> "how impure are the children, adjusted by their size?"

So:

- big bad child → matters a lot
- small bad child → matters less

### Simple example

Left:

* 4 samples → Gini = 0

Right:

* 4 samples → Gini = 0.32

$$Weighted = \frac{4}{8}(0) + \frac{4}{8}(0.32) = 0.16$$


### Key insight

**Gini answers:**

> “How mixed is one group?”

**Weighted Gini answers:**

> “How good is this split overall?”


### One-line summary

```text
Gini = impurity of a node
Weighted Gini = impurity of a split
```

### Gini split procedure:
- Sort feature values
- For each adjacent pair:
    - compute midpoint threshold
- For each threshold:
    - split data
    - compute weighted Gini
- Choose best threshold

## Gini Optimization Algorithm (Step by Step)

### Step 1 - Sort Features

In [66]:
df_sorted = df.sort_values(by='studied')
df_sorted

,studied,passed
0,1,0
1,2,0
2,3,0
3,4,0
4,5,1
5,6,1
6,7,1
7,8,1
8,9,1
9,10,1


### Step 2: Compute Midpoint Threshold For Each Adjacent Pair   

In [67]:
# compute mid points between values of studied
mid_points = []
for i in range(len(df['studied']) - 1):
    mid_point = (df['studied'][i] + df['studied'][i+1]) / 2
    # [i] refer to the fist value and [i+1] refer to the second value and so on
    print(mid_point)
    mid_points.append(mid_point)

1.5
2.5
3.5
4.5
5.5
6.5
7.5
8.5
9.5
10.5
11.5


### Step 4A : Split base on first midpoint threshold

In [68]:
# Split data on first mid point
split_value = mid_points[0]
df_left = df[df['studied'] <= split_value]
df_right = df[df['studied'] > split_value]

print(f"Left split:\n{df_left}\n")
print(f"Right split:\n{df_right}\n")

Left split:
   studied  passed
0        1       0

Right split:
    studied  passed
1         2       0
2         3       0
3         4       0
4         5       1
5         6       1
6         7       1
7         8       1
8         9       1
9        10       1
10       11       1
11       12       1



### Step 4B: Compute Weighted gini 

In [69]:
def gini(y):
    '''
    Compute the Gini impurity for a set of labels.
    y: array-like of shape (n_samples,) - target labels for the samples in the node
    return: float - Gini impurity of the node
    '''
    if len(y) == 0:
        return 0
    p0 = (y == 0).sum() / len(y)
    p1 = (y == 1).sum() / len(y)

    #print(f"p0: {p0:.2f}, p1: {p1:.2f}")

    gini = 1 - (p0**2 + p1**2)
    return gini

In [70]:
# Test gini function
print(f"Gini impurity: {gini(df['passed']):.2f}")

Gini impurity: 0.44


In [71]:
# Compute Gini for the left and right splits
gini_left = gini(df_left['passed'])
gini_right = gini(df_right['passed'])

print(f"Gini impurity for left split: {gini_left:.2f}")
print(f"Gini impurity for right split: {gini_right:.2f}")

Gini impurity for left split: 0.00
Gini impurity for right split: 0.40


In [72]:
# weighted gini impurity
def weighted_gini(df_left, df_right):
    total_samples = len(df_left) + len(df_right)
    gini_left = gini(df_left['passed'])
    gini_right = gini(df_right['passed'])
    
    weighted_gini = (len(df_left) / total_samples) * gini_left + (len(df_right) / total_samples) * gini_right
    return weighted_gini

In [73]:
# For midpoint 1.5
weighted_gini_1_5 = weighted_gini(df_left, df_right)
print(f"Weighted Gini impurity for split at 1.5: {weighted_gini_1_5:.2f}")

Weighted Gini impurity for split at 1.5: 0.36


In [74]:
def best_gini_split(df):
    best_gini = float('inf')
    best_split_value = None
    
    for i in range(len(df['studied']) - 1):
        mid_point = (df['studied'][i] + df['studied'][i+1]) / 2
        df_left = df[df['studied'] <= mid_point]
        df_right = df[df['studied'] > mid_point]
        
        current_gini = weighted_gini(df_left, df_right)
        
        if current_gini < best_gini:
            best_gini = current_gini
            best_split_value = mid_point
            
    return best_split_value, best_gini

In [75]:
best_split_value, best_gini = best_gini_split(df)
print(f"Best split value: {best_split_value}, Best Gini impurity: {best_gini:.2f}")

Best split value: 4.5, Best Gini impurity: 0.00


### Create Decision Tree Function

In [76]:
def built_tree(df):

    if df.empty:
        return None

    # If there's only one row left, return its label
    if len(df) <= 1:
        print(f"Single row subset found with label: {int(df['passed'].iloc[0])}")
        return int(df['passed'].iloc[0])
    
    # If all labels are identical, return that label
    # nunique() returns the number of unique values in the 'passed' column
    if df['passed'].nunique() == 1: 
        print(f"Pure subset found with label: {int(df['passed'].iloc[0])}")
        return int(df['passed'].iloc[0])


    # Find the best split based on Gini impurity
    median_studied, best_gini = best_gini_split(df)
    print(f"Best split value: {median_studied}, Gini after split: {best_gini:.2f}")

    # Split the dataset into two subsets based on the median value of 'studied'
    left_subset = df[df['studied'] <= median_studied]
    right_subset = df[df['studied'] > median_studied]

    print(f"Left subset:\n{left_subset}\n")
    print(f"Right subset:\n{right_subset}\n")

    # Create a dictionary to represent the current node
    clf = {'median_studied': float(median_studied)}

    # Recursively build the left and right subtrees
    clf['left'] = built_tree(left_subset)
    clf['right'] = built_tree(right_subset)

    return clf

In [77]:
def gini(y):
    '''
    Compute the Gini impurity for a set of labels.
    y: array-like of shape (n_samples,) - target labels for the samples in the node
    return: float - Gini impurity of the node
    '''
    if len(y) == 0:
        return 0
    p0 = (y == 0).sum() / len(y)
    p1 = (y == 1).sum() / len(y)

    #print(f"p0: {p0:.2f}, p1: {p1:.2f}")

    gini = 1 - (p0**2 + p1**2)
    return gini

In [78]:
# weighted gini impurity
def weighted_gini(df_left, df_right):
    total_samples = len(df_left) + len(df_right)
    gini_left = gini(df_left['passed'])
    gini_right = gini(df_right['passed'])
    
    weighted_gini = (len(df_left) / total_samples) * gini_left + (len(df_right) / total_samples) * gini_right
    return weighted_gini

In [79]:
def best_gini_split(df):
    best_gini = float('inf')
    best_split_value = None
    df_sorted = df.sort_values(by='studied').reset_index(drop=True)
    
    for i in range(len(df_sorted['studied']) - 1):
        mid_point = (df_sorted['studied'][i] + df_sorted['studied'][i+1]) / 2
        df_left = df_sorted[df_sorted['studied'] <= mid_point]
        df_right = df_sorted[df_sorted['studied'] > mid_point]
        
        current_gini = weighted_gini(df_left, df_right)
        
        print(f"Midpoint: {mid_point}, Weighted Gini: {current_gini:.2f}")  
        
        if current_gini < best_gini:
            best_gini = current_gini
            best_split_value = mid_point
            
    return best_split_value, best_gini

### Build Tree 

We build a tree using current data frame.

In [80]:
df

,studied,passed
0,1,0
1,2,0
2,3,0
3,4,0
4,5,1
5,6,1
6,7,1
7,8,1
8,9,1
9,10,1


In [81]:
clf = built_tree(df)

Midpoint: 1.5, Weighted Gini: 0.36
Midpoint: 2.5, Weighted Gini: 0.27
Midpoint: 3.5, Weighted Gini: 0.15
Midpoint: 4.5, Weighted Gini: 0.00
Midpoint: 5.5, Weighted Gini: 0.13
Midpoint: 6.5, Weighted Gini: 0.22
Midpoint: 7.5, Weighted Gini: 0.29
Midpoint: 8.5, Weighted Gini: 0.33
Midpoint: 9.5, Weighted Gini: 0.37
Midpoint: 10.5, Weighted Gini: 0.40
Midpoint: 11.5, Weighted Gini: 0.42
Best split value: 4.5, Gini after split: 0.00
Left subset:
   studied  passed
0        1       0
1        2       0
2        3       0
3        4       0

Right subset:
    studied  passed
4         5       1
5         6       1
6         7       1
7         8       1
8         9       1
9        10       1
10       11       1
11       12       1

Pure subset found with label: 0
Pure subset found with label: 1


In [82]:
clf


{'median_studied': 4.5, 'left': 0, 'right': 1}

In [83]:
# print tree structure in json format
print(json.dumps(clf, indent=8))


{
        "median_studied": 4.5,
        "left": 0,
        "right": 1
}


In [84]:
# print tree structure with guided lines
def print_tree(node, indent="    "):
    if isinstance(node, dict):
        print(f"{indent}Median studied: {node['median_studied']}")
        print(f"{indent}├─ Left:")
        print_tree(node['left'], indent + "│   ")
        print(f"{indent}└─ Right:")
        print_tree(node['right'], indent + "    ")
    else:
        print(f"{indent}Passed: {node}")


In [85]:
print_tree(clf)

    Median studied: 4.5
    ├─ Left:
    │   Passed: 0
    └─ Right:
        Passed: 1


### Prediction / Inference

To make a prediction we pass the prediction to the tree until we reach a leaf.

In [86]:
# make prediction function
def predict(clf, x):
    node = clf  # start at root of the tree

    # while we are still at a decision node (dictionary)
    while isinstance(node, dict):

        # get the split threshold stored at this node
        threshold = node['median_studied']

        # go left if x is smaller or equal to threshold
        if x <= threshold:
            print(f"Going left: {x} <= {threshold}")
            node = node['left']
        # otherwise go right
        else:
            print(f"Going right: {x} > {threshold}")
            node = node['right']

    # when we reach a leaf (not a dict), return the prediction
    return node

In [88]:
number_of_hours_studied = 6
prediction = predict(clf, number_of_hours_studied)
print(f"Prediction for student who studied {number_of_hours_studied} hours: {prediction}")

Going right: 6 > 4.5
Prediction for student who studied 6 hours: 1


## End